In [1]:
import numpy as np
from datasets import load_dataset
import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_arch_list())  # 应该包含 sm_120

True
['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


## Muti-Header Attention Block

In [3]:
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [4]:
def attention(query, key, value, mask=None, dropout=None):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

In [5]:
class MultiHeadsAttention(nn.Module):

    def __init__(self, d_model, h, dropout=0.1):
        super(MultiHeadsAttention, self).__init__()
        assert d_model % h == 0 # 断言投影矩阵维度可以整除头数
        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h
        self.linears = clones(nn.Linear(d_model, d_model), 4)#QKVO四个线性投影变换
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2) #拆分多头
            for lin, x in zip(self.linears, (query, key, value))
        ]
        print("query:",query.shape)
        print("key:", key.shape  )
        print("value:", value.shape)
        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)
        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.h * self.d_k)
        )
        del query
        del key
        del value
        return self.linears[-1](x) #WO

In [6]:
def test_attention():
    torch.manual_seed(0)

    d_model = 512
    h = 64
    B = 2
    L = 10

    att = MultiHeadsAttention(d_model, h, dropout=0.0)  # dropout=0 方便复现
    att.eval()

    print("=" * 60)
    print(f"配置: d_model={d_model}, h={h}, d_k={d_model//h}")
    print(f"输入: B={B}, L={L}")
    print("=" * 60)

    # 输入必须是 [B, L, d_model]
    x = torch.randn(B, L, d_model)
    print(f"\n输入 x: {tuple(x.shape)}   # [B, L, d_model]")

    # 用自注意力：query = key = value = x
    out = att(x, x, x)
    print(f"输出 out: {tuple(out.shape)}   # [B, L, d_model]")

    assert out.shape == (B, L, d_model), "输出形状不对！"

    # 查看注意力矩阵
    print(f"\nattn: {tuple(att.attn.shape)}   # [B, h, L, L]")

    # 验证每行 softmax 和为 1
    row_sum = att.attn.sum(dim=-1)
    print(f"attn 每行和: min={row_sum.min():.4f}, max={row_sum.max():.4f}")
    assert torch.allclose(row_sum, torch.ones_like(row_sum), atol=1e-5)

In [7]:
if __name__ == "__main__":
    test_attention()

配置: d_model=512, h=64, d_k=8
输入: B=2, L=10

输入 x: (2, 10, 512)   # [B, L, d_model]
query: torch.Size([2, 64, 10, 8])
key: torch.Size([2, 64, 10, 8])
value: torch.Size([2, 64, 10, 8])
输出 out: (2, 10, 512)   # [B, L, d_model]

attn: (2, 64, 10, 10)   # [B, h, L, L]
attn 每行和: min=1.0000, max=1.0000


## 层归一化
    对每个特征维做归一化，然后做仿射变换，仿射变换让网络自行选择进行归一化的程度，否则任意的输入都会被归一化成正态分布

In [8]:
print(nn.Parameter(torch.ones(2, 3)))
print(nn.Parameter(torch.zeros(2, 3)))

Parameter containing:
tensor([[1., 1., 1.],
        [1., 1., 1.]], requires_grad=True)
Parameter containing:
tensor([[0., 0., 0.],
        [0., 0., 0.]], requires_grad=True)


In [9]:
class LayerNorm(nn.Module):

    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

## 残差连接

In [10]:
class SublayerConnection(nn.Module):

    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

## 位置编码

In [11]:
class PositionwiseFeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))

## Encoder

In [12]:
class EncoderLayer(nn.Module):

    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask)) #自注意力塞进残差块
        return self.sublayer[1](x, self.feed_forward)

## Encoder测试程序

In [13]:
class EncoderClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, h, d_ff, n_layers,
                 num_classes, max_len=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_embed = nn.Embedding(max_len, d_model)

        attn = MultiHeadsAttention(d_model, h, dropout)
        ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.layers = clones(EncoderLayer(d_model, attn, ff, dropout), n_layers)

        self.norm = LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)

        # 词嵌入 + 位置嵌入
        x = self.dropout(self.embed(input_ids) * math.sqrt(self.d_model) + self.pos_embed(pos))

        # 构造 mask: [B, 1, 1, L]，用于屏蔽 padding
        mask = attention_mask.unsqueeze(1).unsqueeze(2)  # [B, 1, 1, L]

        for layer in self.layers:
            x = layer(x, mask)

        x = self.norm(x)
        # 用 mask 做平均池化，忽略 padding
        mask_f = attention_mask.unsqueeze(-1).float()     # [B, L, 1]
        x = (x * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)
        return self.classifier(self.dropout(x))

In [14]:
def load_imdb_data(batch_size=32, max_len=200, max_samples=None):

    from datasets import load_dataset
    from transformers import AutoTokenizer

    print("正在加载 IMDB 数据集...")
    dataset = load_dataset("stanfordnlp/imdb")

    if max_samples:
        dataset["train"] = dataset["train"].select(range(max_samples))
        dataset["test"] = dataset["test"].select(range(max_samples // 2))

    print("正在加载 tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_len,
                         padding="max_length")

    dataset = dataset.map(tokenize, batched=True)
    dataset.set_format(type="torch",
                       columns=["input_ids", "attention_mask", "label"])

    train_loader = DataLoader(dataset["train"], batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset["test"], batch_size=batch_size)

    return train_loader, test_loader, tokenizer.vocab_size

In [15]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for step, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)

        if step % 50 == 0:
            print(f"  step {step:4d} | loss {loss.item():.4f}")

    return total_loss / total, correct / total

In [16]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [17]:
def test_encoder_on_imdb():
    torch.manual_seed(0)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"设备: {device}")

    # 超参（小模型，保证能在 CPU 上跑）
    BATCH_SIZE = 32
    MAX_LEN = 128
    D_MODEL = 128
    H = 4
    D_FF = 256
    N_LAYERS = 2
    NUM_CLASSES = 2
    EPOCHS = 3
    LR = 1e-4

    # 加载数据
    train_loader, test_loader, vocab_size = load_imdb_data(
        batch_size=BATCH_SIZE, max_len=MAX_LEN, max_samples=2000
    )
    print(f"训练 batch 数: {len(train_loader)}, 测试 batch 数: {len(test_loader)}")

    # 建模型
    model = EncoderClassifier(
        vocab_size=vocab_size,
        d_model=D_MODEL, h=H, d_ff=D_FF,
        n_layers=N_LAYERS, num_classes=NUM_CLASSES,
        max_len=MAX_LEN, dropout=0.1
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"模型参数量: {n_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    # 训练
    for epoch in range(1, EPOCHS + 1):
        print(f"\n===== Epoch {epoch} =====")
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        print(f"Train: loss={train_loss:.4f}, acc={train_acc:.4f}")
        print(f"Test:  loss={test_loss:.4f}, acc={test_acc:.4f}")

    # 最终断言
    assert test_acc > 0.6, f"测试准确率太低: {test_acc}"
    print(f"\n✅ EncoderLayer 在 IMDB 上测试通过，最终准确率 {test_acc:.4f}")


if __name__ == "__main__":
    test_encoder_on_imdb()

设备: cuda
正在加载 IMDB 数据集...


正在加载 tokenizer...
训练 batch 数: 63, 测试 batch 数: 32
模型参数量: 6,714,754

===== Epoch 1 =====
query: torch.Size([32, 4, 128, 32])
key: torch.Size([32, 4, 128, 32])
value: torch.Size([32, 4, 128, 32])


RuntimeError: The size of tensor a (128) must match the size of tensor b (4096) at non-singleton dimension 1